In [ ]:
from picamera2 import Picamera2
import mediapipe as mp
import cv2
import numpy as np
from PIL import Image
from xgolib import XGO 
import spidev as SPI
import xgoscreen.LCD_2inch as LCD_2inch
from key import Button

In [ ]:
import ipywidgets.widgets as widgets
# 图像数据转化  Image data transformation
def bgr8_to_jpeg(value, quality=75):
    return bytes(cv2.imencode('.jpg', value)[1])

# 创建摄像头显示组件  Create the camera display component
image_widget = widgets.Image(format='jpeg', width=320, height=240) 

In [ ]:
dog = XGO(port='/dev/ttyAMA0',version="xgolite")

mydisplay = LCD_2inch.LCD_2inch()
mydisplay.clear()
splash = Image.new("RGB", (mydisplay.height, mydisplay.width ),"black")
mydisplay.ShowImage(splash)
button=Button()

picam2 = Picamera2()
picam2.configure(
    picam2.create_preview_configuration(main={"format": "RGB888", "size": (320, 240)})
)
picam2.start()
# length=0

mpHands = mp.solutions.hands  
hands = mpHands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

def handDetector(img):
    length = 0
  
    results = hands.process(img)
    
    if results.multi_hand_landmarks:
        for handlms in results.multi_hand_landmarks:
            for index, lm in enumerate(handlms.landmark):
                h, w = img.shape[:2]
                cx, cy = int(lm.x * w), int(lm.y * h)
                if index == 4:
                    x1, y1 = cx, cy
                if index == 8:
                    x2, y2 = cx, cy
                  
                    cv2.circle(img, (x1,y1), 5, (0,0,255), cv2.FILLED) 
                    cv2.circle(img, (x2,y2), 5, (0,0,255), cv2.FILLED) 
                    cv2.line(img, (x1,y1), (x2,y2), (255,0,255), 3) 
                    length = ((x1-x2)**2 + (y1-y2)**2)**0.5
                    length = min(int(length), 1000)
    return img, length



In [ ]:
display(image_widget)
while True:
    img = picam2.capture_array()
    img, length = handDetector(img)
    img_pil = Image.fromarray(img)
    mydisplay.ShowImage(img_pil)

    r,g,b = cv2.split(img)
    frame = cv2.merge((b,g,r))
    image_widget.value = bgr8_to_jpeg(frame)
    # cv2.imshow('frame',frame)
    # cv2.waitKey(1)
    
    print(f"Detected finger distance: {length}")
    
    if length > 0:
        h = min(max(length, 0), 100)
        target_height = 75 + (h / 100 * 40)
        dog.translation('z', target_height)
        print(f"Setting height to: {target_height}")
    else:
        dog.translation('z', 95)
        
    if button.press_b():
        dog.reset()
        break

picam2.stop()
picam2.close()
